# Regional statistical complexity and consciousness recovery

**Public research-code companion.** This notebook reconstructs the statistical-learning workflow from the author’s original analysis, using **private, precomputed regional features**. It does not contain raw EEG, patient-level data, recovery labels, or cached outputs. It cannot run without separately authorized data.

The manuscript reports 20 participants (7 recovered), a fixed depth-3 decision tree, 7-fold stratified cross-validation repeated 1,000 times, 1,000 label permutations, and Benjamini–Hochberg FDR correction within each measure/condition. The analysis is exploratory and lacks external validation.

## 1. Imports and analysis parameters

In [ ]:
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score
from statsmodels.stats.multitest import multipletests

N_REPEATS = 1000
N_PERMUTATIONS = 1000
N_FOLDS = 7
MAX_DEPTH = 3
ALPHA = 0.05
CLINICAL_BENCHMARK = 0.78
REGIONS = ["Occipital", "Parietal", "Temporal", "Frontal", "Central"]


## 2. Load private derived features

Provide a locally stored, institutionally authorized feature table with **one row per participant** and columns `Recovery`, the five named regions, and optionally `Whole`. Do not commit the table or participant identifiers. Regional pairs are constructed as two separate model inputs; pre-averaged pair columns are not used.

The provided spreadsheets are **not** distributed with this repository.

In [ ]:
PRIVATE_FEATURES = Path("private_data/propofol_features.csv")
if not PRIVATE_FEATURES.exists():
    raise FileNotFoundError(
        "Private feature table unavailable. See README for the required schema. "
        "Do not upload confidential participant data to GitHub."
    )
df = pd.read_csv(PRIVATE_FEATURES)
required = ["Recovery", *REGIONS]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df[required].isna().any().any():
    raise ValueError("Missing values in required fields")
if set(df["Recovery"].unique()) != {0, 1}:
    raise ValueError("Recovery must be coded 0/1 with both classes represented")
print(f"Loaded {len(df)} private participant-level feature rows")


## 3. Cross-validation and permutation inference

The functions below retain the original notebook’s **mean of per-fold balanced accuracies**, randomized stratified folds, fixed depth-3 tree, and comparison of the observed mean across repeated CV runs with the permutation null. This is an implementation of the supplied notebook, not a claim of exact numerical reproducibility across environments or random-number streams. Small test folds can contain only one class; scikit-learn may warn about the resulting fold-level balanced accuracy.

In [ ]:
def repeated_fold_balanced_accuracy(X, y, seed):
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    rng = np.random.default_rng(seed)
    scores = []
    for train, test in cv.split(X, y):
        model = DecisionTreeClassifier(max_depth=MAX_DEPTH, criterion="gini",
                                       random_state=int(rng.integers(0, 100_000)))
        model.fit(X[train], y[train])
        scores.append(balanced_accuracy_score(y[test], model.predict(X[test])))
    return float(np.mean(scores))


def cohens_d(a, b):
    pooled = np.sqrt(((len(a)-1)*np.var(a, ddof=1) +
                      (len(b)-1)*np.var(b, ddof=1)) / (len(a)+len(b)-2))
    return float((np.mean(a)-np.mean(b))/pooled) if pooled > 0 else np.nan


def evaluate_features(X, y, seed=2026):
    rng = np.random.default_rng(seed)
    observed = np.array([repeated_fold_balanced_accuracy(X, y, s)
                         for s in range(N_REPEATS)])
    null = np.array([repeated_fold_balanced_accuracy(
        X, rng.permutation(y), s + 100_000)
        for s in range(N_PERMUTATIONS)])
    p_perm = float(np.mean(null >= observed.mean()))
    critical = np.percentile(null, 100*(1-ALPHA))
    return {
        "mean_balanced_accuracy": observed.mean(),
        "ci95_low": np.percentile(observed, 2.5),
        "ci95_high": np.percentile(observed, 97.5),
        "p_perm": p_perm,
        "cohens_d": cohens_d(observed, null),
        "empirical_power_estimate": np.mean(observed > critical),
        "p_clinical_baseline": np.mean(observed <= CLINICAL_BENCHMARK),
    }


## 4. Evaluate 16 spatial configurations

Run separately for each EEG measure and physiological condition. The FDR family is the 16 spatial configurations within a given measure × condition. The complete manuscript considered four measures and two conditions; this notebook demonstrates the pipeline for the provided statistical-complexity feature table.

In [ ]:
y = df["Recovery"].to_numpy(dtype=int)
configurations = {region: [region] for region in REGIONS}
configurations.update({a[0]+b[0]: [a, b] for a, b in combinations(REGIONS, 2)})
if "Whole" in df:
    configurations["WHOLE"] = ["Whole"]

records = []
for label, columns in configurations.items():
    result = evaluate_features(df[columns].to_numpy(dtype=float), y)
    records.append({"feature": label, **result})
results = pd.DataFrame(records)
results["p_perm_fdr"] = multipletests(results["p_perm"], method="fdr_bh")[1]
results["significant_fdr"] = results["p_perm_fdr"] < ALPHA
results.sort_values("p_perm_fdr")


## 5. Clinical benchmark and interpretation

The 78% benchmark is a literature-derived comparison, **not** independently measured clinician performance in this cohort. The original analysis reports empirical proportions of repeated CV scores at or below 0.78, alongside 95% intervals; it is distinct from permutation significance. Do not interpret either as external validation.

**Reported manuscript findings (not generated by this unexecuted public notebook):** Only propofol statistical-complexity parietal–temporal and parietal–frontal pairs survived FDR correction. The parietal–temporal model had permutation and FDR-adjusted *p* < 0.001, a reported 95% accuracy interval of 0.821–0.964, and *p* = 0.011 against the 0.78 benchmark. The parietal–frontal pair had adjusted *p* = 0.032 but did not significantly exceed the benchmark.

**Scope:** The provided feature tables begin *after* raw-EEG preprocessing and complexity extraction. This notebook therefore does not reproduce the EEG signal-processing stage.